# CTSM CONUS Input Data Reference
Documents every category of input file used by the control case (compset `I2000Clm60SpMizGsNldas`), what it contains, its shape/structure, and how it relates to the other files. Grouped by role rather than by CIME's component list (datm/ctsm/cpl/mizuroute), since several files serve the same underlying purpose across components.

In [1]:
import xarray as xr
import pandas as pd

def describe(path, label):
    print(f"\n{'='*90}\n{label}\n{path}\n{'='*90}")
    try:
        ds = xr.open_dataset(path)
    except Exception as e:
        print(f"  Could not open: {e}")
        return None
    print(f"Dimensions: {dict(ds.sizes)}")
    print(f"Data variables ({len(ds.data_vars)}):")
    for v in ds.data_vars:
        print(f"  {v:25s} {str(ds[v].dims):30s} {ds[v].dtype}")
    print(f"Coordinates: {list(ds.coords)}")
    if ds.attrs:
        print(f"Global attrs: {dict(ds.attrs)}")
    return ds

In [6]:
nld_0 = describe("/glade/campaign/ral/hap/common/NLDAS2/1999/1999010100.LDASIN_DOMAIN1", "nldas")



nldas
/glade/campaign/ral/hap/common/NLDAS2/1999/1999010100.LDASIN_DOMAIN1
Dimensions: {'lat_0': 30, 'lon_0': 60, 'Time': 1}
Data variables (12):
  lat                       ('lat_0',)                     float32
  lon                       ('lon_0',)                     float32
  Times                     ()                             |S20
  valid_time                ('Time',)                      datetime64[ns]
  T2D                       ('Time', 'lat_0', 'lon_0')     float64
  Q2D                       ('Time', 'lat_0', 'lon_0')     float64
  U2D                       ('Time', 'lat_0', 'lon_0')     float64
  V2D                       ('Time', 'lat_0', 'lon_0')     float64
  PSFC                      ('Time', 'lat_0', 'lon_0')     float64
  RAINRATE                  ('Time', 'lat_0', 'lon_0')     float64
  SWDOWN                    ('Time', 'lat_0', 'lon_0')     float64
  LWDOWN                    ('Time', 'lat_0', 'lon_0')     float64
Coordinates: ['lat_0', 'lon_0']


## 1. Atmospheric forcing (DATM) -- NLDAS2
Time-varying meteorological data driving the land model. Split into three streams by variable group. All share the same 0.125 deg CONUS grid (224 lat x 464 lon, 25-53N/235-293E) and monthly file cadence (1980-2018).

- **Precip**: `PRECTmms` -- total precipitation rate (mm/s)
- **Solar**: shortwave radiation components
- **TPQWL**: temperature, pressure, humidity, wind, longwave radiation (the remaining atmospheric state variables)

These three streams together provide everything CLM needs to compute the land surface energy/water balance -- this is the data whose Precip stream we found had a near-zero-value artifact at one specific coastal grid cell during spinup validation.

In [2]:
FORCING_ROOT = "/glade/campaign/cesm/cesmdata/inputdata/atm/datm7/atm_forcing.datm7.NLDAS2.0.125d.v1"

ds_precip = describe(f"{FORCING_ROOT}/Precip/ctsmforc.NLDAS2.0.125d.v1.Prec.1985-06.nc", "DATM Precip stream (example month)")
ds_solar  = describe(f"{FORCING_ROOT}/Solar/ctsmforc.NLDAS2.0.125d.v1.Solr.1985-06.nc", "DATM Solar stream (example month)")
ds_tpqwl  = describe(f"{FORCING_ROOT}/TPQWL/ctsmforc.NLDAS2.0.125d.v1.TPQWL.1985-06.nc", "DATM TPQWL stream (example month)")


DATM Precip stream (example month)
/glade/campaign/cesm/cesmdata/inputdata/atm/datm7/atm_forcing.datm7.NLDAS2.0.125d.v1/Precip/ctsmforc.NLDAS2.0.125d.v1.Prec.1985-06.nc
Dimensions: {'lat': 224, 'lon': 464, 'scalar': 1, 'time': 720}
Data variables (7):
  LONGXY                    ('lat', 'lon')                 float32
  LATIXY                    ('lat', 'lon')                 float32
  EDGEE                     ('scalar',)                    float32
  EDGEW                     ('scalar',)                    float32
  EDGES                     ('scalar',)                    float32
  EDGEN                     ('scalar',)                    float32
  PRECTmms                  ('time', 'lat', 'lon')         float32
Coordinates: ['time']
Global attrs: {'case_title': 'NLDAS 1-Hourly Atmospheric Forcing: Precipitation'}

DATM Solar stream (example month)
/glade/campaign/cesm/cesmdata/inputdata/atm/datm7/atm_forcing.datm7.NLDAS2.0.125d.v1/Solar/ctsmforc.NLDAS2.0.125d.v1.Solr.1985-06.nc
Dimens

## 2. Meshes (ESMFmesh) -- "where is everything" spatial definitions
Each mesh file describes the geometry (element centroids and/or polygon vertices) for one specific grid or network, in a format ESMF's regridder can consume. These do NOT contain data values -- only geometry.

- `model_meshfile`/`mesh_atm`/`mesh_lnd`/`mesh_mask`: all point to the SAME file -- the shared 0.125 deg CONUS land/atm grid mesh.
- `mesh` under `atm_forcing...ESMFmesh...`: same grid, mesh copy bundled specifically for the forcing dataset.
- `mesh_rof`: the mizuRoute river-segment mesh (`centerCoords`, `nodeCoords`) -- this is the one used to build the USGS-gauge-to-reachID crosswalk earlier.

In [3]:
MESH_ROOT = "/glade/campaign/cesm/cesmdata/inputdata/share/meshes"
ROF_ROOT = "/glade/campaign/cesm/cesmdata/inputdata/rof/mizuRoute"

ds_mesh_land = describe(f"{MESH_ROOT}/0.125nldas2_ESMFmesh_cd5_241220.nc", "Shared land/atm mesh (model_meshfile / mesh_atm / mesh_lnd / mesh_mask)")
ds_mesh_rof  = describe(f"{ROF_ROOT}/meshes/polygon_centroid/MERITmz_0.125nldas2_ctrcrd_sqr_cdf5_ESMFmesh_c20210113.nc", "mizuRoute river-segment mesh (mesh_rof)")


Shared land/atm mesh (model_meshfile / mesh_atm / mesh_lnd / mesh_mask)
/glade/campaign/cesm/cesmdata/inputdata/share/meshes/0.125nldas2_ESMFmesh_cd5_241220.nc
Dimensions: {'origGridRank': 2, 'nodeCount': 104625, 'coordDim': 2, 'elementCount': 103936, 'maxNodePElement': 4}
Data variables (7):
  origGridDims              ('origGridRank',)              int32
  nodeCoords                ('nodeCount', 'coordDim')      float64
  elementConn               ('elementCount', 'maxNodePElement') float64
  numElementConn            ('elementCount',)              int32
  centerCoords              ('elementCount', 'coordDim')   float64
  elementArea               ('elementCount',)              float64
  elementMask               ('elementCount',)              int32
Coordinates: []
Global attrs: {'gridType': 'unstructured mesh', 'version': '0.9', 'inputFile': 'domain.lnd.0.125nldas2_0.125nldas2.SCRIP.241220.nc', 'timeGenerated': 'Thu Dec 24 14:51:00 2020'}

mizuRoute river-segment mesh (mesh_rof)
/g

## 3. Regridding weight maps -- "how to convert between grids"
Pre-computed area-weighted averaging factors. The mesh files above describe *where* things are; these describe *how much* of each source cell contributes to each destination cell when translating a field from one grid to the other.

- `lnd2rof_map`: converts CLM's gridded runoff -> mizuRoute's river segments (this is how `QRUNOFF`/`basRunoff` gets from the land grid onto the network mizuRoute actually routes).
- `rof2lnd_map`: the reverse direction (used if a field needs to go back onto the land grid, e.g. water table feedback from lakes/reservoirs, if active).

In [4]:
ds_lnd2rof = describe(f"{ROF_ROOT}/gridmaps/map_0.125nldas2_TO_MERITmz_CONUS_aave.210609.nc", "lnd2rof_map (CLM grid -> mizuRoute segments)")
ds_rof2lnd = describe(f"{ROF_ROOT}/gridmaps/map_MERITmz_CONUS_TO_0.125nldas2_aave.210609.nc", "rof2lnd_map (mizuRoute segments -> CLM grid)")


lnd2rof_map (CLM grid -> mizuRoute segments)
/glade/campaign/cesm/cesmdata/inputdata/rof/mizuRoute/gridmaps/map_0.125nldas2_TO_MERITmz_CONUS_aave.210609.nc
Dimensions: {'src_grid_rank': 2, 'dst_grid_rank': 1, 'n_a': 103936, 'n_b': 227247, 'n_s': 664054}
Data variables (13):
  src_grid_dims             ('src_grid_rank',)             int32
  dst_grid_dims             ('dst_grid_rank',)             int32
  xc_a                      ('n_a',)                       float64
  yc_a                      ('n_a',)                       float64
  xc_b                      ('n_b',)                       float64
  yc_b                      ('n_b',)                       float64
  mask_a                    ('n_a',)                       int32
  mask_b                    ('n_b',)                       int32
  frac_a                    ('n_a',)                       float32
  frac_b                    ('n_b',)                       float32
  S                         ('n_s',)                       flo

## 4. River network topology (ntopo)
The actual connectivity graph of the river network mizuRoute routes flow through -- which segment flows into which downstream segment, segment length/slope, and (after your first run) the restart state for channel storage. This is distinct from the mesh file (#2): ntopo defines the network's *plumbing* (routing physics inputs), while the mesh defines the network's *geographic shape* (for regridding/plotting).

In [5]:
ds_ntopo = describe(f"{ROF_ROOT}/ancillary_data/ntopo_MERITmz_0.125nldas2_cdf5_fill_reorder_c20210625.nc", "River network topology (fname_ntopold)")


River network topology (fname_ntopold)
/glade/campaign/cesm/cesmdata/inputdata/rof/mizuRoute/ancillary_data/ntopo_MERITmz_0.125nldas2_cdf5_fill_reorder_c20210625.nc
Dimensions: {'seg': 227247, 'hru': 227247}
Data variables (8):
  length                    ('seg',)                       float64
  seg_id                    ('seg',)                       int32
  Tosegment                 ('seg',)                       int32
  Slope                     ('seg',)                       float64
  pfaf                      ('seg',)                       |S30
  Basin_Area                ('hru',)                       float64
  hru_seg_id                ('hru',)                       int32
  hruid                     ('hru',)                       int32
Coordinates: []
Global attrs: {'history': 'Wed Jun  9 22:10:27 2021: ./subset_river_network.py MERIT_hydro/netcdf/v1/ntopo_MERIT_Hydro_v1_fill_fixed.reorder.nc MERIT_hydro/data/v1/nldas/nldas_MERIT_v1_catch_hruid.csv MERIT_hydro/netcdf/v1/ntopo_M

## 5. Static land surface data (fsurdat)
Time-invariant (for this SP-mode config) land surface characteristics: PFT composition, soil texture/color, irrigated-vs-rainfed crop functional type split (this is where the irrigation config's `irrigate` namelist flag takes effect), lake/wetland/urban fractions, etc. Built specifically for the 0.125 deg CONUS/NLDAS2 grid -- this is the file that does NOT have a corresponding transient (flanduse_timeseries) version, which is why the land-cover config is currently blocked.

In [6]:
ds_fsurdat = describe(
    "/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/surfdata_esmf/ctsm5.4.0/surfdata_0.125nldas2_hist_2000_78pfts_c251022.nc",
    "Static surface dataset (fsurdat)"
)


Static surface dataset (fsurdat)
/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/surfdata_esmf/ctsm5.4.0/surfdata_0.125nldas2_hist_2000_78pfts_c251022.nc
Dimensions: {'lsmlat': 224, 'lsmlon': 464, 'nlevsoi': 10, 'numurbl': 3, 'numrad': 2, 'nlevurb': 10, 'nglcecp1': 11, 'nglcec': 10, 'cft': 64, 'natpft': 15, 'time': 12, 'lsmpft': 79}
Data variables (82):
  LONGXY                    ('lsmlat', 'lsmlon')           float64
  LATIXY                    ('lsmlat', 'lsmlon')           float64
  mxsoil_color              ()                             int32
  SOIL_COLOR                ('lsmlat', 'lsmlon')           int32
  PCT_SAND                  ('nlevsoi', 'lsmlat', 'lsmlon') float32
  PCT_CLAY                  ('nlevsoi', 'lsmlat', 'lsmlon') float32
  mapunits                  ('lsmlat', 'lsmlon')           int32
  ORGANIC                   ('nlevsoi', 'lsmlat', 'lsmlon') float32
  ORGC                      ('nlevsoi', 'lsmlat', 'lsmlon') float32
  BULK                      ('nlevsoi', '

## 6. Global/generic ancillary streams (not CONUS-specific)
These files are built on coarser, GLOBAL grids (f09/0.9x1.25, 0.25deg, etc.) rather than the CONUS 0.125deg grid -- they get regridded on the fly via their own bundled mesh files. They provide inputs CTSM needs regardless of region: snow optics, dust emission roughness, urban building properties, crop calendars, atmospheric chemistry deposition. None of these are specific to the NLDAS2/CONUS setup -- they're the same files any global CTSM run would use.

In [8]:
# PARAM_ROOT = "/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2"

# ds_snicar_optics = describe(f"{PARAM_ROOT}/snicardata/snicar_optics_5bnd_c013122.nc", "Snow optical properties (fsnowoptics)")
# ds_paramfile     = describe(f"{PARAM_ROOT}/paramdata/ctsm60_params.c260518.nc", "CLM60 global parameter file (paramfile)")

## 7. Initial condition (finidat)
Your case's actual starting physical state -- for the current second-pass 38yr run, this is the restart from the first eval run's end date (`control.clm2.r.2018-10-01-00000.nc`), NOT the original f09 CMIP7 restart used for the very first spinup. Check `ctsm.input_data_list` in your active case directory to confirm which restart is currently referenced, since this changes across spinup -> eval run 1 -> eval run 2.

In [9]:
# Path depends on which case's run directory you're checking -- update as needed
FINIDAT_PATH = "/glade/derecho/scratch/mozhgana/control/run/control.clm2.r.2018-10-01-00000.nc"
ds_finidat = describe(FINIDAT_PATH, "Initial condition / restart (finidat)")


Initial condition / restart (finidat)
/glade/derecho/scratch/mozhgana/control/run/control.clm2.r.2018-10-01-00000.nc
Dimensions: {'gridcell': 80439, 'landunit': 210999, 'column': 356231, 'levmaxurbgrnd': 25, 'pft': 611892, 'glc_nec1': 11, 'levsno': 12, 'vegwcs': 4, 'levlak': 10, 'levcan': 1, 'numrad': 2, 'levtot': 37, 'levgrnd': 25, 'levsno1': 13, 'ntapes_multiply_by_max_split_files': 4}
Data variables (292):
  timemgr_rst_nstep_rad_prev ()                             float64
  timemgr_rst_type          ()                             float64
  timemgr_rst_step_sec      ()                             float64
  timemgr_rst_start_ymd     ()                             float64
  timemgr_rst_start_tod     ()                             float64
  timemgr_rst_ref_ymd       ()                             float64
  timemgr_rst_ref_tod       ()                             float64
  timemgr_rst_curr_ymd      ()                             float64
  timemgr_rst_curr_tod      ()                   

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'timemgr_rst_nstep_rad_prev' has multiple fill values {np.int32(-9999), np.int32(-999999999)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'timemgr_rst_type' has multiple fill values {np.int32(-9999), np.int32(-999999999)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'timemgr_rst_step_sec' has multiple fill values {np.int32(-9999), np.int32(-999999999)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: v

## Summary: how these files relate, end to end

```
DATM forcing (Precip/Solar/TPQWL, 0.125deg CONUS grid)
        |  (regridded via model_meshfile/mesh_atm, same grid as land -- no conversion needed here)
        v
CLM land model (uses fsurdat for static surface props, paramfile for physics params,
                finidat for starting state)
        |  produces QRUNOFF/basRunoff on the 0.125deg CONUS grid
        v
lnd2rof_map  (regrids CLM's gridded runoff onto mizuRoute's river segments,
              using mesh_lnd + mesh_rof geometry to know where each segment sits
              relative to each grid cell)
        v
mizuRoute (uses ntopo for network connectivity/routing physics,
           produces RoutedRunoff per reachID)
        |  (optionally regridded back via rof2lnd_map if needed)
        v
Your evaluation: reachID -> USGS gauge crosswalk (flow_multi_agency_all_meta.csv)
                 -> compare RoutedRunoff vs observed streamflow
```
